# Lab: Process JSON Data with Exceptions and Classes

Build a small support-ticket processor. You will parse JSON, handle bad records without crashing, and organise the code with classes.

**How this lab works**

1. Read each task and complete the starter code. Replace `pass` with your code.
2. Run **Check your work** to see which tasks pass.
3. Stuck? Open **Reference solutions** at the end, then try again without looking.

**Skills used:** [JSON](./02_07_Working_With_JSON.ipynb), [exceptions](./02_03_Exception_Handling.ipynb), [classes](./02_05_Classes_And_Objects.ipynb) and [functions](./02_02_Functions_And_Return_Values.ipynb).

## The data

Each ticket must be an object with an integer `id`, a non-empty string `subject`, and a `priority` that is `"low"`, `"medium"` or `"high"`.
The sample below has some good tickets and some bad ones on purpose.

In [ ]:
import json

RAW_TICKETS = """
[
  {"id": 1, "subject": "Cannot log in", "priority": "high"},
  {"id": 2, "subject": "Update billing address", "priority": "low"},
  {"id": 3, "subject": "App crashes on start", "priority": "high"},
  {"id": 4, "subject": "", "priority": "medium"},
  {"id": "5", "subject": "Wrong ID type", "priority": "low"},
  {"id": 6, "subject": "Bad priority", "priority": "urgent"},
  {"subject": "Missing id", "priority": "low"},
  {"id": 8, "subject": "Feature request: dark mode", "priority": "medium"}
]
"""
VALID_PRIORITIES = ("low", "medium", "high")


# Setup: a tiny helper that reports pass/fail without stopping the notebook
def check(label, func, *args, expected):
    try:
        actual = func(*args)
    except Exception as error:
        print(f"✘ {label}: raised {type(error).__name__}: {error}")
        return
    if actual == expected:
        print(f"✔ {label}")
    else:
        print(f"✘ {label}: expected {expected!r}, got {actual!r}")


def check_raises(label, exception_type, func, *args):
    try:
        func(*args)
    except exception_type:
        print(f"✔ {label}")
    except Exception as error:
        print(f"✘ {label}: raised {type(error).__name__}, expected {exception_type.__name__}")
    else:
        print(f"✘ {label}: no exception raised")

## Task 1: Custom exceptions and parsing

- Define `TicketDataError` (parent) and `TicketValidationError` (a child of `TicketDataError`).
- Write `parse_tickets(raw)` that returns the list of ticket dictionaries.
  Raise `TicketDataError` if `raw` is not valid JSON or is not a JSON **array**. Keep the original error with `from`.

In [ ]:
class TicketDataError(Exception):
    pass


class TicketValidationError(TicketDataError):
    pass


def parse_tickets(raw):
    pass

## Task 2: Validate one ticket

Write `validate_ticket(data)` that returns `data` when it is valid and otherwise raises `TicketValidationError` with a helpful message.
A ticket is valid when it is a dictionary with:

- `id`: an `int`
- `subject`: a non-empty string
- `priority`: one of `VALID_PRIORITIES`

In [ ]:
def validate_ticket(data):
    pass

## Task 3: The `Ticket` class

Create a class `Ticket` with:

- `__init__(self, id, subject, priority)` storing the three values
- `is_urgent()` returning `True` when the priority is `"high"`
- `to_dict()` returning the ticket as a dictionary
- `__str__` returning text like `#1 [high] Cannot log in`
- a class method `from_dict(cls, data)` that validates `data` with `validate_ticket` and returns a `Ticket`

In [ ]:
class Ticket:
    pass

## Task 4: The `TicketQueue` class

Create a class `TicketQueue` that holds tickets and offers:

- `add(ticket)`
- `urgent()`: a list of the urgent tickets
- `by_priority()`: a dictionary mapping each priority to the **count** of tickets, for all three priorities (use `0` when none)
- `to_json()`: a JSON string (indented by 2) of the ticket dictionaries
- `__len__`

In [ ]:
class TicketQueue:
    pass

## Task 5: Load safely

Write `load_tickets(raw)` that returns a tuple `(queue, errors)`.

- Parse `raw`, then try to build a `Ticket` from every record and add it to a new `TicketQueue`.
- A bad record must **not** stop the others. Collect a message such as `"record 4: subject must be a non-empty string"` in `errors` (record numbers start at 1).
- If `raw` itself cannot be parsed, let the `TicketDataError` propagate.

In [ ]:
def load_tickets(raw):
    pass

## Check your work

In [ ]:
def run_checks():
    # Task 1
    check("Task 1: parses a list", lambda: len(parse_tickets('[{"id": 1}]')), expected=1)
    check_raises("Task 1: invalid JSON", TicketDataError, parse_tickets, "{oops")
    check_raises("Task 1: not an array", TicketDataError, parse_tickets, '{"id": 1}')

    # Task 2
    good = {"id": 1, "subject": "Help", "priority": "low"}
    check("Task 2: valid ticket", validate_ticket, good, expected=good)
    check_raises("Task 2: empty subject", TicketValidationError, validate_ticket,
                 {"id": 1, "subject": "", "priority": "low"})
    check_raises("Task 2: bad id type", TicketValidationError, validate_ticket,
                 {"id": "1", "subject": "x", "priority": "low"})
    check_raises("Task 2: bad priority", TicketValidationError, validate_ticket,
                 {"id": 1, "subject": "x", "priority": "urgent"})
    check_raises("Task 2: missing key", TicketValidationError, validate_ticket,
                 {"subject": "x", "priority": "low"})

    # Task 3
    check("Task 3: str", lambda: str(Ticket(1, "Cannot log in", "high")), expected="#1 [high] Cannot log in")
    check("Task 3: urgent", lambda: Ticket(1, "x", "high").is_urgent(), expected=True)
    check("Task 3: not urgent", lambda: Ticket(1, "x", "low").is_urgent(), expected=False)
    check("Task 3: from_dict/to_dict", lambda: Ticket.from_dict(good).to_dict(), expected=good)
    check_raises("Task 3: from_dict validates", TicketValidationError,
                 lambda: Ticket.from_dict({"id": 1}))

    # Task 4
    def queue_stats():
        queue = TicketQueue()
        queue.add(Ticket(1, "a", "high"))
        queue.add(Ticket(2, "b", "low"))
        queue.add(Ticket(3, "c", "high"))
        return (len(queue), [t.id for t in queue.urgent()], queue.by_priority(),
                json.loads(queue.to_json())[1])

    check("Task 4: queue", queue_stats,
          expected=(3, [1, 3], {"low": 1, "medium": 0, "high": 2},
                    {"id": 2, "subject": "b", "priority": "low"}))

    # Task 5
    def load_stats():
        queue, errors = load_tickets(RAW_TICKETS)
        return len(queue), len(errors), errors[0]

    check("Task 5: load", load_stats,
          expected=(4, 4, "record 4: subject must be a non-empty string"))
    check_raises("Task 5: unparseable input", TicketDataError, load_tickets, "not json")


run_checks()

## Bonus

1. Add `save(path)` and `load(path)` methods to `TicketQueue` that use a JSON file. Clean up the file afterwards.
2. Print a report: how many tickets loaded, the urgent ones, and every error message.
3. Add a custom exception for duplicate ticket ids and raise it from `TicketQueue.add`.

In [ ]:
# Bonus: your code here

## Reference solutions

Try the tasks first. These cells redefine everything and re-run the checks.

In [ ]:
class TicketDataError(Exception):
    """The ticket data could not be read."""


class TicketValidationError(TicketDataError):
    """A single ticket has invalid content."""


def parse_tickets(raw):
    try:
        data = json.loads(raw)
    except json.JSONDecodeError as error:
        raise TicketDataError(f"invalid JSON: {error.msg}") from error
    if not isinstance(data, list):
        raise TicketDataError("expected a JSON array of tickets")
    return data


def validate_ticket(data):
    if not isinstance(data, dict):
        raise TicketValidationError("ticket must be an object")
    for key in ("id", "subject", "priority"):
        if key not in data:
            raise TicketValidationError(f"missing key {key!r}")
    if not isinstance(data["id"], int) or isinstance(data["id"], bool):
        raise TicketValidationError("id must be an integer")
    if not isinstance(data["subject"], str) or not data["subject"].strip():
        raise TicketValidationError("subject must be a non-empty string")
    if data["priority"] not in VALID_PRIORITIES:
        raise TicketValidationError(f"priority must be one of {VALID_PRIORITIES}")
    return data


class Ticket:
    def __init__(self, id, subject, priority):
        self.id = id
        self.subject = subject
        self.priority = priority

    def is_urgent(self):
        return self.priority == "high"

    def to_dict(self):
        return {"id": self.id, "subject": self.subject, "priority": self.priority}

    def __str__(self):
        return f"#{self.id} [{self.priority}] {self.subject}"

    @classmethod
    def from_dict(cls, data):
        validate_ticket(data)
        return cls(data["id"], data["subject"], data["priority"])


class TicketQueue:
    def __init__(self):
        self._tickets = []

    def add(self, ticket):
        self._tickets.append(ticket)

    def urgent(self):
        return [t for t in self._tickets if t.is_urgent()]

    def by_priority(self):
        counts = {priority: 0 for priority in VALID_PRIORITIES}
        for ticket in self._tickets:
            counts[ticket.priority] += 1
        return counts

    def to_json(self):
        return json.dumps([t.to_dict() for t in self._tickets], indent=2)

    def __len__(self):
        return len(self._tickets)


def load_tickets(raw):
    queue = TicketQueue()
    errors = []
    for number, record in enumerate(parse_tickets(raw), start=1):
        try:
            queue.add(Ticket.from_dict(record))
        except TicketValidationError as error:
            errors.append(f"record {number}: {error}")
    return queue, errors


run_checks()

Finally, see the result of the whole pipeline.

In [ ]:
queue, errors = load_tickets(RAW_TICKETS)

print(f"Loaded {len(queue)} tickets; {len(errors)} rejected")
print("Urgent:", [str(t) for t in queue.urgent()])
print("By priority:", queue.by_priority())
print("Errors:")
for message in errors:
    print(" -", message)